## Chapter 3 — Qiskit add-ons for error mitigation

Welcome to chapter 3! In this chapter, we will use the tools introduced in chapter 2 to implement advanced _per-layer_ error mitigation techniques. Let's review the story so far: Chapter 1 reviewed the mitigation techniques available as `Estimator` options (`DD`, `PT`, `TREX`, `ZNE`, `PEA` and `PEC`), these are _whole-circuit switches_ (`PEC` and `PEA` do per-layer mitigation *internally*, but the API does not surface per-layer knobs). Chapter 2 introduced Samplomatic allowing us to box circuits and localize error mitigation decisions to specific layers. Chapter 2 also showed us noise learning.

This chapter combines those ingredients into structure-aware add-ons that go beyond whole-circuit techniques. We focus on three techniques (two Qiskit add-ons):

- **Section 3.1: Propagated noise absorption (PNA)** — mitigates gate noise by rewriting the observable, leaving the circuit itself unchanged.
- **Section 3.2: Probabilistic error cancellation (PEC)** — mitigates errors by sampling from inverse-noise rewrites of the circuit.
- **Section 3.2: Shaded lightcones (SLC)** — applies PEC's idea selectively, mitigating only what can affect the observable to reduce sampling overhead.

_PNA_ and _SLC_ are the main focus of this lab — these are the *new tools for quantum advantage* the title of the lab points to. By acting per layer, absorbing or cancelling each layer's *own* learned noise, they reach past what the Runtime `Estimator` options can do, and keep the sampling overhead low enough to stay practical as circuits grow.

We will explore error mitigation techniques using the same 1D transverse-field Ising mirror circuit as in Chapter 2. We reuse the `construct_ising_circuit` function from Section 2.6.1 and the mirror trick but scale it up to a longer chain so the per-layer effects are larger and we can see the impact of the error mitiagtion techniques when we implement them.

## 3.1 Propagated noise absorption (PNA)

This section introduces **[Propagated Noise Absorption (PNA)](https://qiskit.github.io/qiskit-addon-pna/)**, an error mitigation technique that inverts learned noise and absorbs it into the observable rather than into the circuit itself. Where traditional probabilistic error cancellation (PEC) requires the sampling of many variants of noisy circuits (as we shall see in section 3.2), PNA sidesteps that overhead and exploits the structure we have built up in Chapters 1 and 2: dressed layers, Pauli propagation through Cliffords, boxed circuits with `Twirl` and `InjectNoise` annotations, and noise learned as a sparse Pauli-Lindblad generator.

The key insight is that when noise is Pauli (or can be twirled into Pauli form), its inverse can be **propagated forward** through the circuit using the Clifford-conjugation rules from section 1.1.3. The noise can then **absorbed into the measurement observable** to produce a _noise mitigating observable_.

Because Pauli channels compose and propagate efficiently, PNA computes a single noise mitigating observable $\tilde{O}$ that, when measured on the noisy circuit, yields an estimate of the noise-free expectation value of the original desired observable $\langle O \rangle$. 

This section walks through the four-step PNA workflow: boxing, learning, propagating, and absorbing. We use the Ising mirror circuit from Chapter 2 as our example and scale it up to 10 qubits with 2 Trotter steps, performing the entire PNA workflow for this example.


### 3.1.1 Set-up and scaling up to 10 qubits

In this section, we continue working with the __1D transverse-field Ising chain__ Hamiltonian introduced in section 2.6:
$$
H = -J \sum_{\langle i,j \rangle} Z_i Z_j + h \sum_i X_i \;.
$$


We scale up to 10 qubits, keeping 2 Trotter steps and rotation angle π/8. Below, we simply set up the problem: we construct the Ising mirror circuit and transpile.

In [ ]:
# create 10 qubit settings
# ── Circuit ─────────────────────────────────────────────────────────
num_qubits_pna          = 10           # chain length
num_trotter_steps_pna   = 2            # Trotter "reps"
rx_angle           = np.pi / 8    # transverse-field rotation per step

# create the Ising chain circuit
ising_pna  = construct_ising_circuit(num_qubits_pna, num_trotter_steps_pna, rx_angle)
mirror_pna = ising_pna.compose(ising_pna.inverse())
mirror_pna.measure_all()

# transpile to obey ISA
mirror_isa_pna = isa_pm.run(mirror_pna)
layout_pna     = mirror_isa_pna.layout.final_index_layout()

print(f"Mirror     : {num_qubits_pna}q × {num_trotter_steps_pna} Trotter steps, "
      f"rx_angle = π/{round(np.pi / rx_angle)}")
print(f"ISA layout : physical qubits {layout_pna}")

### 3.1.2 Boxing the circuit and learning the noise

Before we can box the new Ising mirror circuit, we need to create a new boxing pass manager for PNA. Chapter 2 used a boxing pass manager with `inject_noise_strategy="no_modification"`, which fed the learned noise model into the samplex as a passthrough but never applied it to the sampled circuits. We need different strategies in Chapter 3:

- **PNA** uses `inject_noise_strategy="uniform_modification"`: every layer's noise is modified the _same way_ at sampling time. PNA sets `noise_scales` to 0 on all `ref`'s, which leaves the sampled circuits untouched while still associating each layer with its learned model so the noise model can be propagated into the observable.
- **SLC** uses `inject_noise_strategy="individual_modification"`: each layer (and each generator) is modified independently, so SLC can scale noise per generator along the observable's lightcone. We will see this in section 3.2.

For more information on the choice of `inject_noise_strategy`, see the [documentation](https://qiskit.github.io/samplomatic/api/auto/samplomatic.transpiler.generate_boxing_pass_manager.html#samplomatic.transpiler.generate_boxing_pass_manager/inject_noise_strategy) for `generate_boxing_pass_manager`.

After defining the new boxing pass manager, we find the unique layers in the circuit using `find_unique_box_instructions`. We print the number of unique layers below.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PNA — boxing manager for uniform_modification + find unique layers
# ═══════════════════════════════════════════════════════════════════
pna_boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    measure_annotations="all",
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="uniform_modification",
)

# box the circuit for pna
boxed_circuit_pna = pna_boxing_pm.run(mirror_isa_pna)
# find the unique layers in the circuit
unique_layers_pna = find_unique_box_instructions(
    boxed_circuit_pna, normalize_annotations=None, undress_boxes=True,
)
print(f"PNA boxed  : {len(unique_layers_pna)} unique layers")

Now we are ready to learn the noise in the circuit. We need to this again here for the 10-qubit Ising circuit since this circuit is different to the circuits we learned the noise for previously. Noise in a quantum computer changes over time (calibration happens everyday), so cached learning results from a previous calibration window should be re-learned in general.

First, we initialize the noise learner options:

In [ ]:
# ── NoiseLearnerV3 new options (used by PNA, SLC, and indirectly by PEC) ───────
NL_NUM_RANDOMIZATIONS  = 16
NL_SHOTS_PER_RAND      = 32
NL_LAYER_PAIR_DEPTHS   = [1, 2, 4, 8]

pna_nl_options = NoiseLearnerV3Options(
    num_randomizations=NL_NUM_RANDOMIZATIONS,
    shots_per_randomization=NL_SHOTS_PER_RAND,
    layer_pair_depths=NL_LAYER_PAIR_DEPTHS,
)
pna_learner = NoiseLearnerV3(backend, pna_nl_options)

_Note:_ The following cell executes a job on quantum hardware. Ensure you are ready to do this before proceeding. After first execution, we recommend pasting the job id into the `NOISE_LEARN_JOB_ID_PNA` parameter and setting `SUBMIT_NOISE_JOB_PNA = False`. This will prevent accidentally resubmitting the job and keeps the notebook re-runnable after a kernel restart.

_Estimated QPU execution time is 15 seconds (tested on ibm_pittsburgh)_. The usage estimate above reflects backend execution time only. Queue time, calibration, and runtime session delays may be longer.

In [ ]:
NOISE_LEARN_JOB_ID_PNA = None # paste job_id here on re-run
SUBMIT_NOISE_JOB_PNA   = False   # set True to submit a fresh learning job
pna_learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_PNA is not None:
    learner_job_pna = service.job(NOISE_LEARN_JOB_ID_PNA)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_PNA}")
elif SUBMIT_NOISE_JOB_PNA:
    learner_job_pna = pna_learner.run(unique_layers_pna)
    NOISE_LEARN_JOB_ID_PNA = learner_job_pna.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_PNA}")
else:
    print("Set SUBMIT_NOISE_JOB_PNA=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_PNA and re-run.")


In [ ]:
learner_job_pna = service.job(NOISE_LEARN_JOB_ID_PNA)
print(f"{NOISE_LEARN_JOB_ID_PNA}  (status: {learner_job_pna.status()})")

In [ ]:
if learner_job_pna.status() == "DONE":
    noise_learner_result_pna = learner_job_pna.result()
else:
    print(f"Not done yet (status={learner_job_pna.status()}). Re-run cell when DONE.")

Extract the dict `refs_to_noise_models_pna` containing the learned noise in each unique layer:

In [ ]:
if 'noise_learner_result_pna' in dir() and noise_learner_result_pna is not None:
    refs_to_noise_models_pna = noise_learner_result_pna.to_dict(unique_layers_pna, require_refs=False)
    print(f"refs_to_noise_models_pna has {len(refs_to_noise_models_pna)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


### The observable of interest

Part of the transverse-field Ising Hamiltonian is a ZZ-interaction energy term. For our observable of interest, we consider one such term, on the middle two qubits:

$$
O \;=\;  Z_{4} Z_{5}, \;
$$
for $N=10$ qubits. This is a nearest-neighbor ZZ correlator. On the mirror circuit's ideal output $|0\rangle^{\otimes N}$, every $\langle Z_i Z_{i+1}\rangle = +1$, so the ideal expectation value is $\langle O\rangle = 1$.

PNA rewrites the observable $O$ into an modified _noise mitigating_ observable $\tilde{O}$. This modified observable will contain additional terms with different coefficients. The idea is that the noisy expectation value of $\tilde{O}$ matches the ideal expectation of the original $O$. 

First, we construct the observable $O$ and then apply the circuits's layout to it.

In [ ]:
# create observable for the Ising model
# Single ZZ on the middle two qubits (1 term, ideal exp val = 1).
mid = num_qubits_pna // 2
observable_pna =  SparsePauliOp.from_sparse_list(
    [("ZZ", [mid - 1, mid], 1.0)], num_qubits=num_qubits_pna)

print(observable_pna)

In [ ]:
# apply layout to observable
observable_pna_isa = observable_pna.apply_layout(mirror_isa_pna.layout)

### 3.1.3  Computing the noise mitigating observable $\tilde{O}$

Now we have constructed the observable $O$, the next step in the PNA workflow is to calculate the noise mitigating observable $\tilde{O}$ using the function [`generate_noise_mitigating_observable`](https://qiskit.github.io/qiskit-addon-pna/apidocs/qiskit_addon_pna.html#qiskit_addon_pna.generate_noise_mitigating_observable) from the PNA add-on. The function takes in the boxed circuit, the observable and the learned noise dict. 

Each `InjectNoise(ref=...)` annotation knows its layer's `PauliLindbladMap` from the noise learning step. PNA propagates the inverse of those channels through the circuit and folds them into the observable. The result is a new observable $\tilde{O}$ whose noisy expectation equals the ideal expectation of the original $O$.

A truncation step keeps only the dominant terms so $\tilde{O}$ remains measurable. The more terms an observable has the more measurements of the circuit required to estimate its expectation value. The truncation is controlled by `max_err_terms` and `max_obs_terms`.

The boxed circuit itself is unchanged; only the observable changes.

In [ ]:
# PNA parameters
num_processes = 8
max_err_terms = 10000
max_obs_terms = 10000

# generate the noise mitigating observable
obs_tilde_isa = generate_noise_mitigating_observable(
    boxed_circuit_pna,
    observable_pna_isa,
    refs_to_noise_models_pna,
    max_err_terms=max_err_terms,
    max_obs_terms=max_obs_terms,
    num_processes=num_processes,
    print_progress=True,
    search_step=8,
)


In [ ]:
print(f"Number of Pauli terms in the noise mitigating observable: {len(obs_tilde_isa)}")

We need to do some post-processing to build the noise mitigating observable $\tilde{O}$ in logical qubit space. This is useful so we can inspect the observable and see what terms it contains. 

In [ ]:
### Mapping the noise mitigating observable from physical to logical qubits

# Create an inverse layout: physical qubit index → logical qubit index
physical_to_logical = {
    physical_q: logical_q
    for logical_q, physical_q in enumerate(layout_pna)
}

# Extract the noise mitigating observable in sparse (Pauli, qubits, coeff) form
isa_pauli_terms = obs_tilde_isa.to_sparse_list()

# Remap each Pauli term from physical qubits to logical qubits
logical_pauli_terms = [
    (
        pauli_string,                                  # Pauli operators (e.g. "ZIX")
        [physical_to_logical[q] for q in phys_qubits], # physical → logical indices
        coefficient                                    # real-valued coefficient
    )
    for (pauli_string, phys_qubits, coefficient) in isa_pauli_terms
]

# Rebuild the observable in the logical (virtual) qubit space
obs_tilde_virtual = SparsePauliOp.from_sparse_list(
    logical_pauli_terms,
    num_qubits=num_qubits_pna
)

Below we plot the distribution of magnitudes of every term in $\tilde{O}$.

In [ ]:
# sort the observable magnitudes in descending order for plotting
obs_tilde_virtual = obs_tilde_virtual[np.argsort(np.abs(obs_tilde_virtual.coeffs))][::-1]

plt.figure(figsize=(8, 5))
plt.plot(np.abs(obs_tilde_virtual.coeffs), "o", color="tab:blue", markersize=6, alpha=0.7)
plt.yscale("log")
plt.xlabel("Pauli term index", fontsize=14)
plt.ylabel("Magnitude", fontsize=14)
plt.title(r"$\tilde{O}$ coefficient magnitudes", fontsize=14)
plt.grid(axis="both", alpha=0.3)
plt.tight_layout()
plt.show()

We can see that $\tilde{O}$ has many terms, lots of which are very low in magnitude. Let's sort by magnitude and show the first 10 terms to inspect the structure. Remember that the original observable had only 1 term (ZZ on the central two qubits).

In [ ]:
### Truncating the noise mitigating observable
num_terms_to_plot = 10

# Take the absolute value of all coefficients
coeff_magnitudes = np.abs(obs_tilde_virtual.coeffs)

# Get indices that sort coefficients from smallest to largest
sorted_indices = np.argsort(coeff_magnitudes)

# Reverse the order to place largest coefficients first, reorder using sorted indices
sorted_indices_desc = sorted_indices[::-1]
obs_tilde_virtual = obs_tilde_virtual[sorted_indices_desc]

# Keep only the most significant Pauli terms
obs_tilde_virtual = obs_tilde_virtual[:num_terms_to_plot]


In [ ]:
def full_pauli_string(pstr, qubits, n):
    full = ["I"] * n
    for p, q in zip(pstr, qubits):
        full[q] = p
    return "".join(reversed(full))   # qubit 0 leftmost

# Sort by |coefficient| descending and keep top 10.
sorted_idx = np.argsort(np.abs(obs_tilde_virtual.coeffs))[::-1][:10]
obs_tilde_top = obs_tilde_virtual[sorted_idx]

# Identify which top-10 terms are originals vs PNA-added.
target_paulis_virtual = {
    full_pauli_string(p, q, obs_tilde_virtual.num_qubits)
    for p, q, _ in observable_pna.to_sparse_list()
}
labels = [
    full_pauli_string(p, q, obs_tilde_virtual.num_qubits)
    for p, q, _ in obs_tilde_top.to_sparse_list()
]
coeffs = np.abs(obs_tilde_top.coeffs)
is_orig = np.array([l in target_paulis_virtual for l in labels])

# Plot.
fig, ax = plt.subplots(figsize=(10, 4))
idx = np.arange(len(labels))
ax.plot(idx[is_orig],  coeffs[is_orig],  "s", color="C3", markersize=12, label="Original ZZ term")
ax.plot(idx[~is_orig], coeffs[~is_orig], "o", color="C0", markersize=10, label="PNA-added")
ax.set_yscale("log")
ax.set_xticks(idx)
ax.set_xticklabels(labels, rotation=45, ha="right", family="monospace")
ax.set_xlabel("Pauli term (virtual qubits)")
ax.set_ylabel(r"$|\tilde{c}|$")
ax.set_title(r"Top 10 terms in $\tilde{O}_{Z}$")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

You should see the original ZZ term has the largest magnitude. The additional terms are accounting for noise in the circuit and so we expect them to be much smaller in magnitude. 

Let's directly compare the magnitudes of terms in the original observable $O$ and the noise mitigating observable $\tilde{O}$.

In [ ]:
# Extract Pauli strings and coefficients
pauli_strings = observable_pna.paulis.to_labels()
coeffs = observable_pna.coeffs

# Sort by coefficient magnitude
sorted_indices = sorted(
    range(len(coeffs)),
    key=lambda i: abs(coeffs[i]),
    reverse=True
)

# Print header
print("\n Original Observable Pauli Terms:\n")
print(f"{'Pauli string':<{observable_pna.num_qubits + 2}}  Coefficient")
print("-" * (observable_pna.num_qubits + 20))

# Print terms 
for i in sorted_indices[:num_terms_to_plot]:
    print(f"{pauli_strings[i]}  {coeffs[i].real:+.3f}")


In [ ]:
num_terms_to_plot = 10 

# Extract Pauli strings and coefficients
pauli_strings = obs_tilde_virtual.paulis.to_labels()
coeffs = obs_tilde_virtual.coeffs

# Sort by coefficient magnitude
sorted_indices = sorted(
    range(len(coeffs)),
    key=lambda i: abs(coeffs[i]),
    reverse=True
)

# Print header
print("\n Noise mitigating Observable Pauli Terms (top 10):\n")
print(f"{'Pauli string':<{obs_tilde_virtual.num_qubits + 2}}  Coefficient")
print("-" * (obs_tilde_virtual.num_qubits + 20))

# Print terms (no divider)
for i in sorted_indices[:num_terms_to_plot]:
    print(f"{pauli_strings[i]}  {coeffs[i].real:+.3f}")

The middle ZZ term had magnitude 1 in the original observable. In $\tilde{O}$ it is slightly above 1 and remains the dominant term: PNA amplified the original term and added new ones to absorb the per-layer noise. The table above shows the 10 largest terms.

Your results will vary depending on the QPU you are using and the noise learned from the QPU. Noise drifts and calibrations happen daily. Even for the same QPU, the noise will change over time. For the most accurate results, you should always relearn your noise just before running the final job.

### Exercise 4 — Mitigate the magnetization observable

<div class="alert alert-block alert-success">

<b>Exercise 4 — Mitigate the magnetization observable</b>

In this exercise you need to build the PNA noise mitigating observable of the **magnetization** observable for a 10-qubit Ising chain:

$$
O_Z = \sum_{i=0}^{N-1} Z_i, \;
$$

the sum of a $Z$ on each qubit. On the mirror's ideal output $|0\rangle^{\otimes N}$ every $\langle Z_i \rangle = +1$, so the ideal expectation value is $\langle O_Z \rangle = 10$ for the 10-qubit chain.

PNA absorbs each layer's learned noise into the observable instead of the circuit. Here, you will build $\tilde{O}_Z$, the noise mitigating magnetization observable, on the 10-qubit boxed mirror, then look at the anti-noise terms PNA propagated into it.

Build the noise mitigating observable in three steps:

- **Target:** define `target_observable_ex4` as a `SparsePauliOp` for $O_Z = \sum_{i=0}^{9} Z_i$ on 10 qubits, using `SparsePauliOp.from_sparse_list` with one entry per qubit.
- **Layout:** PNA matches noise layers by physical qubit index, so ISA-map the observable with `apply_layout(mirror_isa_pna.layout)` → `target_observable_ex4_isa`.
- **Mitigate:** call `generate_noise_mitigating_observable` on the ISA observable, passing the pre-set truncation parameters from the stub as keyword arguments → `obs_tilde_ex4`.

The grader checks the target and the noise mitigating observable.

</div>

In [ ]:
#TODO: Add Your code here and set the missing values.


target_observable_ex4     = None
target_observable_ex4_isa = None
obs_tilde_ex4             = None

# PNA parameters (same as in section 3.1.3 main text).
num_processes = 8
max_err_terms = 10000
max_obs_terms = 10000

# 1. Build target_observable_ex4 as a SparsePauliOp with 10 Z terms one
#    on each qubit
target_observable_ex4 = ...

# 2. Apply mirrored_isa_pna.layout to the target so PNA receives an ISA-mapped observable.
target_observable_ex4_isa = ...

# 3. Call generate_noise_mitigating_observable(boxed_circuit_pna, target_observable_ex4_isa,
#    refs_to_noise_models_pna, max_err_terms=..., max_obs_terms=..., num_processes=...).
obs_tilde_ex4 = ...

####  What does $\tilde{O}_{Z}$ look like?

The plot below shows the 20 largest terms of $\tilde{O}_{Z}$ with their Pauli strings on virtual qubits. You should see the original 10 terms (red squares) sit at the top, slightly amplified. The blue circles are the anti-noise corrections that PNA propagated from the learned noise model.


In [ ]:
# Map obs_tilde back to virtual qubits for readable Pauli labels.
p_2_v = {p: v for v, p in enumerate(layout_pna)}
obs_tilde_virtual_ex4 = SparsePauliOp.from_sparse_list(
    [(p, [p_2_v[q] for q in qs], c) for p, qs, c in obs_tilde_ex4.to_sparse_list()],
    num_qubits=num_qubits_pna,
)

# Sort by |coefficient| descending and keep top 20.
sorted_idx = np.argsort(np.abs(obs_tilde_virtual_ex4.coeffs))[::-1][:20]
obs_tilde_top = obs_tilde_virtual_ex4[sorted_idx]

def full_pauli_string(pstr, qubits, n):
    full = ["I"] * n
    for p, q in zip(pstr, qubits):
        full[q] = p
    return "".join(reversed(full))   # qubit 0 leftmost


# Identify which top-20 terms are originals vs PNA-added.
target_paulis_virtual = {
    full_pauli_string(p, q, num_qubits_pna)
    for p, q, _ in target_observable_ex4.to_sparse_list()
}
labels = [full_pauli_string(p, q, num_qubits_pna)
          for p, q, _ in obs_tilde_top.to_sparse_list()]
coeffs = np.abs(obs_tilde_top.coeffs)
is_orig = np.array([l in target_paulis_virtual for l in labels])

# Plot.
fig, ax = plt.subplots(figsize=(10, 4))
idx = np.arange(len(labels))
ax.plot(idx[is_orig],  coeffs[is_orig],  "s", color="C3", markersize=12, label="Original observable terms")
ax.plot(idx[~is_orig], coeffs[~is_orig], "o", color="C0", markersize=10, label="PNA-added")
ax.set_yscale("log")
ax.set_xticks(idx)
ax.set_xticklabels(labels, rotation=45, ha="right", family="monospace")
ax.set_xlabel("Pauli term (virtual qubits)")
ax.set_ylabel(r"$|\tilde{c}|$")
ax.set_title(r"Top 20 terms in $\tilde{O}$")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### 3.1.4  Running on hardware

With $\tilde{O}$ ready, we build the Executor program and submit.

This section requires:
- `boxed_circuit_pna` (with uniform_modification)
- `obs_tilde_ex4` from Exercise 4 (for the magnetization observable)
- `refs_to_noise_models_pna` from the noise learner for 10 qubits 

In section 3.1.5, we will compare the expectation value results for three cases: the unmitigated observable, the noise mitigating observable from the PNA workflow, and the noise mitigation observable from PNA + TREX.

In [ ]:
# Build template circuit and samplex for use with the "Executor"
template_circuit_pna, samplex_pna = samplomatic.build(boxed_circuit_pna)

In [ ]:
# number of terms in noise mitigating observable to truncate to
num_to_measure = 10

# Identify measured qubits in canonical (measurement) order.
meas_box = boxed_circuit_pna.data[-1]
canonical_qubits = [
    i for i, q in enumerate(boxed_circuit_pna.qubits)
    if q in meas_box.qubits
]

# Build qubit index mappings: canonical → physical → virtual (logical).
canonical_to_physical = dict(enumerate(canonical_qubits))
physical_to_virtual   = {p: v for v, p in enumerate(layout_pna)}
canonical_to_virtual  = {
    c: physical_to_virtual[p]
    for c, p in canonical_to_physical.items()
}

# Discover the ChangeBasis ref from the measurement box rather than hard-coding
# the value below (samplomatic auto-assigns refs at boxing time and the value
# depends on the circuit — e.g. "basis2" in some configurations, "basis0" in
# others). The dynamic lookup keeps the rest of this section circuit-agnostic.
change_basis_ann = get_annotation(meas_box.operation, ChangeBasis)
if change_basis_ann is None:
    raise RuntimeError(
        "No ChangeBasis annotation on the measurement box. "
        "The boxing pass manager must be built with measure_annotations='all'."
    )
basis_ref = change_basis_ann.ref
print(f"ChangeBasis ref: {basis_ref!r}")

# Truncate Õ_Z to the largest terms before measurement (lower variance per shot).
obs_tilde_virtual_ex4 = obs_tilde_virtual_ex4[
    np.argsort(np.abs(obs_tilde_virtual_ex4.coeffs))[::-1]
][:num_to_measure]

# Bases needed to cover every Pauli term in the truncated Õ_Z.
meas_bases, bases_reverser = get_measurement_bases(obs_tilde_virtual_ex4)
meas_bases_canonical = [
    np.array(
        [base[canonical_to_virtual[c]] for c in sorted(canonical_to_virtual)],
        dtype=np.uint8,
    )
    for base in meas_bases
]

# Encoding inside the canonical arrays: I=0, Z=1, X=2, Y=3.
# Always include the all-Z basis so we can also extract an *unmitigated* <O_Z>
# from the same execution. The truncated Õ_Z may not naturally need all-Z
# (e.g. if PNA-added X/Y terms dominate the truncation), but adding it costs
# only one extra basis worth of shots and keeps 3.1.5 self-contained.
all_z_canonical = np.ones(num_qubits_pna, dtype=np.uint8)

if any(np.array_equal(b, all_z_canonical) for b in meas_bases_canonical):
    all_z_idx = next(
        i for i, b in enumerate(meas_bases_canonical)
        if np.array_equal(b, all_z_canonical)
    )
    print(f"all-Z basis already present at index {all_z_idx}")
else:
    meas_bases_canonical.append(all_z_canonical)
    all_z_idx = len(meas_bases_canonical) - 1
    print(f"Appended all-Z basis at index {all_z_idx}")

# Decoded bases printout (qubit 0 leftmost) — useful for sanity checks.
def _decode(b):
    return "".join("IZXY"[v] for v in b)

print(f"\nMeasurement bases (canonical order, qubit 0 leftmost):")
for i, b in enumerate(meas_bases_canonical):
    note = "  ← unmitigated baseline" if i == all_z_idx else ""
    print(f"  [{i}] {_decode(b)}{note}")


#### Aside: term count vs. measurement-basis count

The truncation above keeps a fixed *number* of terms, but the quantity that drives shot cost is the number of distinct **measurement bases**. `get_measurement_bases` groups qubit-wise commuting Pauli terms so they share a basis and therefore share shots. A PNA-added term that is diagonal in a basis we already measure can be kept at no extra *basis* cost.

The cell below rebuilds the full $\tilde{O}_Z$ and counts how many terms just past the `num_to_measure` cut fall into the already-required basis set. Keeping those is "free" in measurement-basis terms — though not entirely free statistically: every extra term still contributes its own coefficient and shot noise to the summed estimate of $\langle\tilde{O}_Z\rangle$. The saving is that no new basis, and hence no new shot allocation, is required.


In [ ]:
# ── Aside: term count vs. measurement-basis count ───────────────────────────────
#
# Rebuild the *full* O_Z-tilde (obs_tilde_virtual_ex4 was truncated in place above)
# so we can sweep past the cut and see which extra terms are "free".
p_2_v = {p: v for v, p in enumerate(layout_pna)}
obs_tilde_virtual_full = SparsePauliOp.from_sparse_list(
    [(p, [p_2_v[q] for q in qs], c) for p, qs, c in obs_tilde_ex4.to_sparse_list()],
    num_qubits=num_qubits_pna,
)
obs_full_sorted = obs_tilde_virtual_full[
    np.argsort(np.abs(obs_tilde_virtual_full.coeffs))[::-1]
]

# Bases required by the current truncation.
bases_cut, _ = get_measurement_bases(obs_full_sorted[:num_to_measure])
n_bases_cut  = len(bases_cut)
print(f"Truncation : {num_to_measure} terms  ->  {n_bases_cut} measurement bases")

# Walk the terms just past the cut: a term is "free" if it needs no new basis.
free = 0
for k in range(num_to_measure + 1, len(obs_full_sorted) + 1):
    bases_k, _ = get_measurement_bases(obs_full_sorted[:k])
    if len(bases_k) == n_bases_cut:
        free += 1
    else:
        break  # first term that would require a new basis

if free:
    print(f"Past cut   : {free} further term(s) fit the existing {n_bases_cut} "
          f"bases -- keeping {num_to_measure + free} terms costs no extra basis.")
else:
    print("Past cut   : the next term needs a new basis -- no free terms here.")
print("(Free in *basis* cost; each extra term still adds its own shot noise.)")


In [ ]:
# Control the # of shots during execution
shots_per_randomization_exec = 64
num_randomizations_exec = 1024

# Zero out the noise so it isn't actually injected during execution.
# We only added InjectNoise annotations so PNA could associate the noise
# to layers in the circuit.
samplex_inputs_pna = {f"noise_scales.{ref}": 0.0 for ref in refs_to_noise_models_pna}
samplex_inputs_pna |= {"pauli_lindblad_maps": refs_to_noise_models_pna}

# Specify the bases to measure.
# The ref name "basis2" is what `samplomatic.build` auto-assigned to the
# measurement-box ChangeBasis annotation for this particular circuit.
# To confirm the ref name for any other circuit, run:
#     print(samplex_pna.inputs().make_broadcastable().describe())
# and look for the line beginning with "basis_changes.<ref>".

bases_broadcastable = np.expand_dims(np.array(meas_bases_canonical), axis=1)

#added to accommodate auto generated basis_changes
change_basis_ann = get_annotation(meas_box.operation, ChangeBasis)
basis_ref = change_basis_ann.ref

samplex_inputs_pna |= {"basis_changes": {basis_ref: bases_broadcastable}}

In [ ]:
# Convert samplex_inputs into a dict to pass to QuantumProgram.
samplex_arguments_pna = samplex_pna.inputs().make_broadcastable().bind(**samplex_inputs_pna)

# Instantiate the QuantumProgram with the specified parameters.
program = QuantumProgram(shots=shots_per_randomization_exec)
program.append_samplex_item(
    circuit=template_circuit_pna,
    samplex=samplex_pna,
    samplex_arguments=samplex_arguments_pna,
    shape=(num_randomizations_exec,),
)

#### Run the Executor job

_Note:_ The following cell executes a job on quantum hardware. Ensure you are ready to do this before proceeding.After first execution, we recommend pasting the job id into the `EXECUTOR_JOB_ID_PNA` parameter and setting  `SUBMIT_EXECUTOR_JOB_PNA = False`. This will prevent accidentally resubmitting the job and keeps the notebook re-runnable after a kernel restart.

_Estimated QPU execution time is 30 seconds (tested on ibm_fez)._ The usage estimate reflects backend execution time only. Queue time, calibration, and runtime session delays may be longer.

In [ ]:
executor = Executor(backend)

EXECUTOR_JOB_ID_PNA      = None      # paste an Executor job_id here on re-run
SUBMIT_EXECUTOR_JOB_PNA  = False    # set True to submit a fresh Executor job
executor.options.environment.job_tags = ["qgss26"]

if EXECUTOR_JOB_ID_PNA is not None:
    job_exec_pna = service.job(EXECUTOR_JOB_ID_PNA)
    print(f"Re-using saved job: {EXECUTOR_JOB_ID_PNA}")
elif SUBMIT_EXECUTOR_JOB_PNA:
    job_exec_pna = executor.run(program)
    EXECUTOR_JOB_ID_PNA = job_exec_pna.job_id()      
    print(f"Submitted: {EXECUTOR_JOB_ID_PNA}")
else:
    print("Set SUBMIT_EXECUTOR_JOB_PNA=True to submit a fresh job, "
          "or paste a saved Executor job id into EXECUTOR_JOB_ID_PNA and re-run.")

In [ ]:
# Verify the job type and status
if EXECUTOR_JOB_ID_PNA is not None:
    print(f"Program id  : {job_exec_pna.job_id()}")
    print(f"Status      : {job_exec_pna.status()}")

Extract the job results:

In [ ]:
if job_exec_pna.status() == "DONE":
    exec_results_pna = job_exec_pna.result()
else:
    print(f"Not done yet (status={job_exec_pna.status()}). Re-run cell when DONE.")

#### Adding the TREX error mitigation technique

In sections 1.1.1 and 1.1.2, we introduced [Twirled readout error extinction (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) as an error mitigation technique that can be switched on easily with the `Estimator` primitive. TREX mitigates read-out errors from measurements by randomly inserting an $X$ gate before a measurement and flipping the outcome classically. This diagonalizes the readout error matrix, allowing the read-out error to be mitigated. 

Here, in the PNA workflow, we can implement TREX easily using the function `trex_factors` from the [Qiskit add-ons utils library](https://github.com/Qiskit/qiskit-addon-utils). 

In [ ]:
# Computing the TREX factors
measurement_noise_map = noise_learner_result_pna[2].to_pauli_lindblad_map()
trex_rescale_factors = trex_factors(measurement_noise_map, bases_reverser)

### 3.1.5  Analyzing results

In this section, we compare three estimates of the magnetization expectation: unmitigated, PNA-only, and PNA + TREX. The ideal value of the expectation value on the mirror`s output state $|0\rangle^{\otimes N}$ equals $N=10$. 

We use the function `executor_expectation_values` from the Qiskit add-on utils library to calculate the expectation value in all three cases. 

Let`s begin with the expectation value that is error mitigated with PNA alone.

In [ ]:
# Expectation value with PNA only.
# `bases_reverser` was constructed from `obs_tilde_virtual_ex4`, so it pairs
# each measurement basis to the (post-PNA) Pauli terms diagonal in that basis.
#
# The data axis-0 length is len(meas_bases_canonical), which may be one larger
# than len(bases_reverser) — we appended an all-Z basis for the unmitigated
# baseline. Slice the data to the original PNA bases before passing in.

n_pna_bases = len(bases_reverser)

results_pna = executor_expectation_values(
    exec_results_pna[0]["meas"][:n_pna_bases],
    bases_reverser,
    meas_basis_axis=0,
    avg_axis=1,
    measurement_flips=exec_results_pna[0]["measurement_flips.meas"][:n_pna_bases],
    pauli_signs=None,  # noise_scales=0 ⇒ no PEC ⇒ no Pauli signs
    rescale_factors=None,
)

exp_val_pna = results_pna[0][0]
std_pna     = results_pna[0][1]
print(f"PNA only    : <O_Z> = {exp_val_pna:+.4f}  std = {std_pna:.4e}")


Next, we calculate the noisy unmitigated expectation value:

In [ ]:
# ─── Unmitigated expectation value ───
# We added the all-Z basis to meas_bases_canonical above (index `all_z_idx`),
# so the executor measured it as part of the same job. Slice that subset of
# the data and pair it with the original observable O_Z.

meas_z  = exec_results_pna[0]["meas"][all_z_idx]
flips_z = exec_results_pna[0]["measurement_flips.meas"][all_z_idx]

bases_reverser_unmit = {Pauli("Z" * num_qubits_pna): [target_observable_ex4]}

res_unmit = executor_expectation_values(
    meas_z,
    bases_reverser_unmit,
    meas_basis_axis=None,   # already sliced — no basis axis remaining
    avg_axis=0,              # average over randomizations (now axis 0)
    measurement_flips=flips_z,
    pauli_signs=None,        # noise_scales=0 ⇒ no PEC ⇒ no Pauli signs
    rescale_factors=None,
)
exp_val_unmit, std_unmit = res_unmit[0][0], res_unmit[0][1]
print(f"Unmitigated : <O_Z> = {exp_val_unmit:+.4f}  std = {std_unmit:.4e}")


Finally, we calculate the expectation value with both error mitigation techniques PNA and TREX:

In [ ]:
# ─── PNA + TREX ───
# TREX rescales the data per-basis to absorb the readout (measurement) noise
# learned by NoiseLearnerV3's measurement-layer model. As above, slice the
# data to the original PNA bases (skipping the appended all-Z basis used only
# for the unmitigated baseline).
measurement_noise_map = noise_learner_result_pna[2].to_pauli_lindblad_map()
trex_rescale_factors  = trex_factors(measurement_noise_map, bases_reverser)

n_pna_bases = len(bases_reverser)

results_pna_trex = executor_expectation_values(
    exec_results_pna[0]["meas"][:n_pna_bases],
    bases_reverser,
    meas_basis_axis=0,
    avg_axis=1,
    measurement_flips=exec_results_pna[0]["measurement_flips.meas"][:n_pna_bases],
    pauli_signs=None,  # noise_scales=0 ⇒ no PEC ⇒ no Pauli signs
    rescale_factors=trex_rescale_factors,
)

exp_val_pna_trex = results_pna_trex[0][0]
std_pna_trex     = results_pna_trex[0][1]
print(f"PNA + TREX  : <O_Z> = {exp_val_pna_trex:+.4f}  std = {std_pna_trex:.4e}")

Plot the results:

In [ ]:
# Compare: Unmitigated, PNA, PNA+TREX vs. ideal (= N = 10 for the 10q chain).
experiments = ["Unmitigated", "PNA", "PNA+TREX"]
colors      = ["tab:gray", "tab:blue", "tab:orange"]
markers     = ["o", "s", "^"]

evs  = [exp_val_unmit, exp_val_pna, exp_val_pna_trex]
errs = [std_unmit,     std_pna,     std_pna_trex]
x    = np.arange(len(experiments))

plt.figure(figsize=(6, 4))
for xi, yi, ei, label, color, marker in zip(x, evs, errs, experiments, colors, markers):
    plt.errorbar(
        xi, yi, yerr=ei,
        color=color, marker=marker, markersize=12,
        linestyle="none", capsize=5,
        label=label, zorder=3,
    )

plt.axhline(y=num_qubits_pna, color="green", linestyle="--", linewidth=2,
            label=f"Ideal = N = {num_qubits_pna}", zorder=2)

plt.xticks(x, experiments)
plt.ylabel("Expectation value", fontsize=14)
plt.title(r"10 qubit Ising chain, 2 Trotter steps, $O_Z$ obs",
          fontsize=14)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In the plot above, You should see a clear difference in results for the three implementations. The unmitigated expectation value should be the worst (i.e. the furthest from the ideal value of 10). Implementing PNA by measuring the expectation value of the noise mitigating observable $\tilde{O}$ instead of the unmitigated observable $O$ should improve the result. Finally, by further adding TREX to mitigate read-out errors as well as PNA, we should see the result improve even further. 

This concludes section 3.1 which covered propagated noise absorption (PNA). In section 3.2, we turn to PEC and SLC.